# AI Text Detection Model Evaluation

This notebook evaluates multiple models on a test dataset and compares their performance.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import sys
import time
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

sys.path.append('../')

from models.numpyModels.logistic_regression import LogisticRegression
from models.numpyModels.dnn.neuralnet import NeuralNetwork
from models.numpyModels.rnn.rnn import RNN
from models.numpyModels.rnn.optimizers import AdamOptimizer

os.makedirs('results', exist_ok=True)

## Load Dataset

In [ ]:
inputs_path = "../datasets/clean/teacher_inputs.csv"
outputs_path = "../datasets/clean/teacher_outputs.csv"

def load_csv(file_path):
    try:
        return pd.read_csv(file_path)
    except Exception as e:
        print(f"Error parsing as comma-separated: {str(e)}")
        try:
            return pd.read_csv(file_path, sep='\t')
        except Exception as e2:
            print(f"Error parsing as tab-separated: {str(e2)}")
            try:
                return pd.read_csv(file_path, encoding='latin1')
            except Exception as e3:
                print(f"Error with latin1 encoding: {str(e3)}")
                raise Exception(f"Failed to load CSV file: {file_path}")

print("Loading input dataset...")
inputs_df = load_csv(inputs_path)

print("Loading output dataset...")
outputs_df = load_csv(outputs_path)

print("Input Dataset:")
display(inputs_df.head())
print(f"Input dataset columns: {inputs_df.columns.tolist()}")

print("\nOutput Dataset:")
display(outputs_df.head())
print(f"Output dataset columns: {outputs_df.columns.tolist()}")

def identify_columns(df):
    columns = df.columns.tolist()
    
    id_col = None
    for col in columns:
        if col.lower() == 'id' or 'id' in col.lower():
            id_col = col
            break
    
    if id_col is None and len(columns) > 0:
        id_col = columns[0]
        print(f"No ID column found, using first column: {id_col}")
    
    content_col = None
    for col in columns:
        if col.lower() in ['text', 'content', 'label', 'class']:
            content_col = col
            break
    
    if content_col is None and len(columns) > 1:
        content_col = columns[1]
        print(f"No content column found, using second column: {content_col}")
    
    return id_col, content_col

input_id_col, input_text_col = identify_columns(inputs_df)
output_id_col, output_label_col = identify_columns(outputs_df)

print(f"\nUsing columns - Input: ID={input_id_col}, Text={input_text_col}")
print(f"Using columns - Output: ID={output_id_col}, Label={output_label_col}")

inputs_df = inputs_df.rename(columns={input_id_col: 'ID', input_text_col: 'Text'})
outputs_df = outputs_df.rename(columns={output_id_col: 'ID', output_label_col: 'Label'})

print("\nAfter renaming:")
print(f"Input dataset columns: {inputs_df.columns.tolist()}")
print(f"Output dataset columns: {outputs_df.columns.tolist()}")

## Load Models

In [ ]:
class DatasetWrapper:
    def __init__(self, X, y=None):
        self.X = X
        self.y = y

def load_model(model_type):
    if model_type == "dnn":
        model = NeuralNetwork()
        model.load("../trained_models/numpy/dnn_weights.npz")
        
        with open("../preprocessed/vectorizer.pkl", "rb") as f:
            vectorizer = pickle.load(f)
        
        return model, vectorizer, None, None
    
    elif model_type == "rnn":
        embedding_matrix = np.load("../preprocessed/embedding_matrix.npy")
        
        model = RNN(n_units=64, embedding_matrix=embedding_matrix)
        model.initialize(AdamOptimizer())
        model.load("../trained_models/numpy/rnn_weights.npz")
        
        with open("../preprocessed/word_to_idx.pkl", "rb") as f:
            word_to_idx = pickle.load(f)
        
        return model, None, word_to_idx, embedding_matrix
    
    else:
        model = LogisticRegression()
        model.load("../trained_models/numpy/logistic_weights.npz")
        
        with open("../preprocessed/vectorizer.pkl", "rb") as f:
            vectorizer = pickle.load(f)
        
        return model, vectorizer, None, None

models = {}
vectorizers = {}
word_to_idxs = {}
embedding_matrices = {}

for model_type in ["logistic", "dnn", "rnn"]:
    try:
        model, vectorizer, word_to_idx, embedding_matrix = load_model(model_type)
        models[model_type] = model
        vectorizers[model_type] = vectorizer
        word_to_idxs[model_type] = word_to_idx
        embedding_matrices[model_type] = embedding_matrix
        print(f"Successfully loaded {model_type} model")
    except Exception as e:
        print(f"Error loading {model_type} model: {str(e)}")

## Define Prediction Function

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

def predict_text(model, vectorizer, word_to_idx, embedding_matrix, text, model_type):
    tokens = clean_text(text)
    
    if model_type == "rnn":
        max_seq_length = 100
        sequence = np.zeros((1, max_seq_length), dtype=int)
        
        for j, word in enumerate(tokens[:max_seq_length]):
            if word in word_to_idx:
                sequence[0, j] = word_to_idx[word]
        
        predictions = model.forward_propagation(sequence)
        probability = predictions[0, -1, 0]
    
    else:
        joined_text = " ".join(tokens)
        X_new = vectorizer.transform([joined_text]).toarray()
        
        if model_type == "dnn":
            class DatasetWrapper:
                def __init__(self, X):
                    self.X = X
            probability = model.predict(DatasetWrapper(X_new))[0][0]
        else:
            probability = model.predict_proba(X_new)[0]

    prediction = "AI" if probability >= 0.5 else "Human"
    return prediction, probability

## Make Predictions

In [ ]:
results = {}
for model_type in models.keys():
    results[model_type] = pd.DataFrame(columns=['ID', 'Label'])

total_rows = len(inputs_df)
print(f"Making predictions on {total_rows} samples...")

start_time = time.time()
update_interval = max(1, total_rows // 20)

for index, row in inputs_df.iterrows():
    if index % update_interval == 0 or index == total_rows - 1:
        elapsed = time.time() - start_time
        progress = (index + 1) / total_rows * 100
        print(f"Progress: {progress:.1f}% ({index+1}/{total_rows}) - Elapsed: {elapsed:.1f}s")
    
    id_val = row['ID']
    text = row['Text']
    
    for model_type in models.keys():
        try:
            model = models[model_type]
            vectorizer = vectorizers[model_type]
            word_to_idx = word_to_idxs[model_type]
            embedding_matrix = embedding_matrices[model_type]
            
            prediction, _ = predict_text(model, vectorizer, word_to_idx, embedding_matrix, text, model_type)
            new_row = pd.DataFrame({'ID': [id_val], 'Label': [prediction]})
            results[model_type] = pd.concat([results[model_type], new_row], ignore_index=True)
        except Exception as e:
            print(f"Error with {model_type} model on text {id_val}: {str(e)}")
            new_row = pd.DataFrame({'ID': [id_val], 'Label': ['Error']})
            results[model_type] = pd.concat([results[model_type], new_row], ignore_index=True)

print(f"Predictions completed in {time.time() - start_time:.1f} seconds")

## Save Results to CSV

In [ ]:
for model_type, df in results.items():
    output_path = f"results/{model_type}_predictions.csv"
    df.to_csv(output_path, sep='\t', index=False)
    print(f"Saved predictions for {model_type} model to {output_path}")

## Evaluate Model Performance

In [ ]:
outputs_df[['ID', 'Label']] = outputs_df['ID'].str.split('\t', expand=True)

print("\nAfter splitting 'ID' column:")
print(f"Output dataset columns: {outputs_df.columns.tolist()}")

def calculate_accuracy(predictions_df, ground_truth_df):
    try:
        for df, name in [(predictions_df, 'predictions'), (ground_truth_df, 'ground truth')]:
            if 'ID' not in df.columns:
                raise KeyError(f"'ID' column not found in {name} dataframe")
            if 'Label' not in df.columns:
                raise KeyError(f"'Label' column not found in {name} dataframe")
        
        merged_df = predictions_df.merge(ground_truth_df, on='ID', suffixes=('_pred', '_true'))
        
        correct = (merged_df['Label_pred'] == merged_df['Label_true']).sum()
        total = len(merged_df)
        accuracy = correct / total if total > 0 else 0
        
        return accuracy, correct, total
    except Exception as e:
        print(f"Error calculating accuracy: {str(e)}")
        print(f"Predictions columns: {predictions_df.columns.tolist()}")
        print(f"Ground truth columns: {ground_truth_df.columns.tolist()}")
        return 0, 0, 0

print("\nModel Performance:")
print("-" * 50)
for model_type, df in results.items():
    accuracy, correct, total = calculate_accuracy(df, outputs_df)
    print(f"{model_type.upper()} Model:")
    print(f"  Accuracy: {accuracy:.4f} ({correct}/{total})")
    print(f"  Percentage: {accuracy * 100:.2f}%")
    print("-" * 50)

## Detailed Analysis

In [ ]:
if len(results) > 1:
    try:
        combined_df = outputs_df[['ID', 'Label']].rename(columns={'Label': 'True_Label'})
        
        for model_type, df in results.items():
            combined_df = combined_df.merge(df, on='ID')
            combined_df = combined_df.rename(columns={'Label': f'{model_type}_pred'})
        
        pred_columns = [col for col in combined_df.columns if col.endswith('_pred')]
        combined_df['all_agree'] = combined_df[pred_columns].apply(lambda row: len(set(row)) == 1, axis=1)
        combined_df['all_correct'] = combined_df.apply(
            lambda row: row['all_agree'] and row[pred_columns[0]] == row['True_Label'], 
            axis=1
        )
        
        total = len(combined_df)
        agree_count = combined_df['all_agree'].sum()
        all_correct_count = combined_df['all_correct'].sum()
        
        print("\nModel Agreement Analysis:")
        print(f"Total samples: {total}")
        print(f"All models agree: {agree_count} ({agree_count/total*100:.2f}%)")
        print(f"All models correct: {all_correct_count} ({all_correct_count/total*100:.2f}%)")
        
        disagreement_df = combined_df[~combined_df['all_agree']]
        print(f"\nSamples where models disagree: {len(disagreement_df)}")
        display(disagreement_df.head(10))
    except Exception as e:
        print(f"Error in model agreement analysis: {str(e)}")